# Autonomous Agent Benchmark Harness for Engineering Systems

This notebook demonstrates automated evaluation of AI coding and reasoning agents on multi-step engineering tasks:
1. Symbolic equivalence verification via SymPy (eliminating string matching false negatives).
2. Dimensional analysis and physical unit consistency verification.
3. Step-by-step milestone hallucination localization.
4. Telemetry tracking (pass@k, token overhead, compute efficiency).

In [ ]:
from agent_benchmark.symbolic import SymbolicVerifier, parse_symbolic_expression
from agent_benchmark.dimensional import DimensionalChecker, DimensionVector
from agent_benchmark.harness import get_standard_engineering_suite, BenchmarkEvaluator, AgentProblemAttempt, AgentStepResponse
from agent_benchmark.telemetry import compute_benchmark_summary, BenchmarkReporter

## 1. Symbolic Equivalence vs. Brittle String Matching

In [ ]:
gt = "(P * L**3) / (3 * E * I)"
candidate = "(1/3) * (P / E) * (L**3 / I)"

print("Exact String Match :", gt == candidate)  # False!
print("Symbolic Equivalence:", SymbolicVerifier.verify_equivalence(candidate, gt))  # True!

## 2. Dimensional Homogeneity Verification

In [ ]:
sym_dims = {
    "P": "force",
    "L": "length",
    "E": "pressure",
    "I": DimensionVector(l=4),
}

# Deflection formula correctly yields length [L]
dim_res = DimensionalChecker.infer_dimension("(P * L**3) / (3 * E * I)", sym_dims)
print("Inferred Dimension:", dim_res.to_symbolic_string())

# Physical violation: adding force and velocity
try:
    DimensionalChecker.infer_dimension("P + L", sym_dims)
except Exception as e:
    print("Caught Dimensional Violation:", e)

## 3. Running Multi-Step Engineering Benchmark

In [ ]:
suite = get_standard_engineering_suite()
prob = suite[0]  # Structural beam deflection

attempt = AgentProblemAttempt(
    problem_id="ENG-STR-01",
    total_reasoning_tokens=520,
    step_responses=[
        AgentStepResponse(step_number=1, candidate_symbolic="P * L", candidate_numeric=5000.0),
        AgentStepResponse(step_number=2, candidate_symbolic="(P * L * c) / I"),
        AgentStepResponse(step_number=3, candidate_symbolic="(1/3) * (P / E) * (L**3 / I)", candidate_numeric=0.0125),
    ],
)

eval_res = BenchmarkEvaluator.evaluate_attempt(prob, attempt)
print(f"All steps correct: {eval_res.all_steps_correct} | Hallucinations: {eval_res.hallucination_count}")
for s in eval_res.step_results:
    print(f"  Step {s.step_number}: {s.error_category} (Symbolic={s.symbolic_correct}, Dim={s.dimensional_correct})")